# Feature Engineering – Technical Indicators

Computes technical-analysis features for each stock from raw OHLCV data.

**Features (per ticker):** SMA(10,20), EMA(10,20), RSI(14), MACD(12,26,9) + signal + histogram, Daily Return, Volatility(20) + annualized.

**Tickers:** AAPL, MSFT, GOOGL, AMZN, TSLA, NVDA, META

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load Stock Price Data

Reuses the loader from notebook 01 (handles standard and yfinance multi-header CSVs).

In [2]:
RAW_DIR = Path('../data/raw/prices')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']

def load_stock_csv(file_path):
    """Load stock CSV handling standard and yfinance multi-header formats."""
    first_row = pd.read_csv(file_path, nrows=0)
    if 'Date' in first_row.columns:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
    elif 'Price' in first_row.columns:
        df = pd.read_csv(file_path, header=[0, 1], index_col=0)
        df = df.dropna(how='all')
        df.index = pd.to_datetime(df.index)
        df.index.name = 'Date'
        df.columns = df.columns.droplevel(1)
        df = df.reset_index()
        for col in ['Close', 'High', 'Low', 'Open', 'Volume']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        raise ValueError(f'Unknown CSV format in {file_path}')
    return df.sort_values('Date').reset_index(drop=True)

stock_data = {}
for ticker in TICKERS:
    fp = RAW_DIR / f'{ticker}.csv'
    if fp.exists():
        df = load_stock_csv(fp)
        stock_data[ticker] = df
        print(f'{ticker}: {len(df)} rows ({df["Date"].min().date()} to {df["Date"].max().date()})')
    else:
        print(f'{ticker}: File not found')

print(f'\nTotal tickers loaded: {len(stock_data)}')

AAPL: 66 rows (2020-03-09 to 2020-06-10)
MSFT: 1253 rows (2021-07-30 to 2026-07-28)
GOOGL: 473 rows (2018-07-25 to 2020-06-10)
AMZN: 32 rows (2020-04-27 to 2020-06-10)
TSLA: 239 rows (2019-07-01 to 2020-06-10)
NVDA: 2334 rows (2011-03-03 to 2020-06-10)
META: 1253 rows (2021-07-30 to 2026-07-28)

Total tickers loaded: 7


## 3. Indicator Functions

Implemented with pandas so the logic is transparent (no external TA library).

In [3]:
def add_sma(df, price_col='Close', windows=(10, 20)):
    for w in windows:
        df[f'SMA_{w}'] = df[price_col].rolling(window=w).mean()
    return df

def add_ema(df, price_col='Close', windows=(10, 20)):
    for w in windows:
        df[f'EMA_{w}'] = df[price_col].ewm(span=w, adjust=False).mean()
    return df

def add_rsi(df, price_col='Close', period=14):
    delta = df[price_col].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    df[f'RSI_{period}'] = 100 - (100 / (1 + rs))
    return df

def add_macd(df, price_col='Close', fast=12, slow=26, signal=9):
    ema_fast = df[price_col].ewm(span=fast, adjust=False).mean()
    ema_slow = df[price_col].ewm(span=slow, adjust=False).mean()
    df['MACD'] = ema_fast - ema_slow
    df['MACD_signal'] = df['MACD'].ewm(span=signal, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']
    return df

def add_daily_return(df, price_col='Close'):
    df['Daily_Return'] = df[price_col].pct_change()
    return df

def add_volatility(df, ret_col='Daily_Return', window=20, trading_days=252):
    df[f'Volatility_{window}'] = df[ret_col].rolling(window=window).std()
    df[f'Volatility_{window}_annualized'] = df[f'Volatility_{window}'] * np.sqrt(trading_days)
    return df

def add_all_features(df):
    df = df.copy()
    df = add_daily_return(df)
    df = add_sma(df)
    df = add_ema(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_volatility(df)
    return df

print('Indicator functions defined.')

Indicator functions defined.


## 4. Compute Features for Every Ticker

In [4]:
featured_data = {}
for ticker, df in stock_data.items():
    featured_data[ticker] = add_all_features(df)
    print(f'{ticker}: {featured_data[ticker].shape[1]} columns, {len(featured_data[ticker])} rows')

sample_ticker = TICKERS[0]
print(f'\nFeature columns for {sample_ticker}:')
print(list(featured_data[sample_ticker].columns))
featured_data[sample_ticker].tail()

AAPL: 17 columns, 66 rows
MSFT: 17 columns, 1253 rows
GOOGL: 17 columns, 473 rows
AMZN: 17 columns, 32 rows
TSLA: 17 columns, 239 rows
NVDA: 17 columns, 2334 rows
META: 17 columns, 1253 rows

Feature columns for AAPL:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'SMA_10', 'SMA_20', 'EMA_10', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist', 'Volatility_20', 'Volatility_20_annualized']


,Date,Open,High,Low,Close,Volume,Daily_Return,SMA_10,SMA_20,EMA_10,EMA_20,RSI_14,MACD,MACD_signal,MACD_hist,Volatility_20,Volatility_20_annualized
61,2020-06-04,78.5209,78.8186,77.6471,78.0198,87560400,-0.0086,77.4437,76.3823,77.3870,75.8392,67.9518,2.2593,2.4334,-0.1741,0.0113,0.1786
62,2020-06-05,78.2691,80.3024,78.2401,80.2419,137250400,0.0285,77.7983,76.7282,77.9061,76.2585,73.3006,2.3417,2.4151,-0.0734,0.0125,0.1981
63,2020-06-08,79.9393,80.7502,79.2301,80.7163,95654400,0.0059,78.1510,77.0106,78.4170,76.6830,74.2873,2.4174,2.4155,0.0019,0.0116,0.1849
64,2020-06-09,80.3968,83.6573,80.3653,83.2652,147712400,0.0316,78.8108,77.3613,79.2985,77.3099,78.8168,2.6525,2.4629,0.1896,0.0130,0.2059
65,2020-06-10,84.2116,85.8745,83.7735,85.4074,166651600,0.0257,79.6515,77.8627,80.4092,78.0811,81.7298,2.9774,2.5658,0.4116,0.0132,0.2100


## 5. Verify No Missing Values

SMA(20), RSI(14) and Volatility(20) create NaNs for the initial warm-up rows. We drop those rows so the engineered feature columns have no missing values.

In [5]:
feature_cols = ['SMA_10', 'SMA_20', 'EMA_10', 'EMA_20', 'RSI_14',
                'MACD', 'MACD_signal', 'MACD_hist',
                'Daily_Return', 'Volatility_20', 'Volatility_20_annualized']

print('Missing values in feature columns BEFORE dropping warm-up rows:')
for ticker, df in featured_data.items():
    print(f'  {ticker}: {int(df[feature_cols].isnull().sum().sum())} NaNs')

for ticker in featured_data:
    featured_data[ticker] = featured_data[ticker].dropna(subset=feature_cols).reset_index(drop=True)

print('\nMissing values in feature columns AFTER dropping warm-up rows:')
for ticker, df in featured_data.items():
    print(f'  {ticker}: {int(df[feature_cols].isnull().sum().sum())} NaNs, {len(df)} rows remaining')

Missing values in feature columns BEFORE dropping warm-up rows:
  AAPL: 83 NaNs
  MSFT: 83 NaNs
  GOOGL: 83 NaNs
  AMZN: 83 NaNs
  TSLA: 83 NaNs
  NVDA: 83 NaNs
  META: 83 NaNs

Missing values in feature columns AFTER dropping warm-up rows:
  AAPL: 0 NaNs, 46 rows remaining
  MSFT: 0 NaNs, 1233 rows remaining
  GOOGL: 0 NaNs, 453 rows remaining
  AMZN: 0 NaNs, 12 rows remaining
  TSLA: 0 NaNs, 219 rows remaining
  NVDA: 0 NaNs, 2314 rows remaining
  META: 0 NaNs, 1233 rows remaining


## 6. Save Featured Datasets

One CSV per ticker plus a combined long-format file for downstream modeling.

In [ ]:
combined = []
for ticker, df in featured_data.items():
    out = df.copy()
    out.insert(1, 'Ticker', ticker)
    out.to_csv(PROCESSED_DIR / f'{ticker}_prices_features.csv', index=False)
    combined.append(out)

combined_df = pd.concat(combined, ignore_index=True)
combined_df.to_csv(PROCESSED_DIR / 'features_all_tickers.csv', index=False)
print(f'Saved per-ticker feature files and combined dataset: {combined_df.shape}')
combined_df.head()

Saved per-ticker feature files and combined dataset: (5510, 18)


,Date,Ticker,Open,High,Low,Close,Volume,Daily_Return,SMA_10,SMA_20,EMA_10,EMA_20,RSI_14,MACD,MACD_signal,MACD_hist,Volatility_20,Volatility_20_annualized
0,2020-04-06,AAPL,60.5681,63.5157,60.2012,63.3612,201820400,0.0872,60.2881,60.6466,60.2730,60.7457,69.2712,-0.7962,-1.0905,0.2943,0.0650,1.0323
1,2020-04-07,AAPL,65.3721,65.5893,62.5235,62.6273,202887200,-0.0116,60.5911,60.3339,60.7010,60.9249,68.0575,-0.5526,-0.9829,0.4303,0.0629,0.9985
2,2020-04-08,AAPL,63.4263,64.5440,63.0618,64.2302,168895200,0.0256,61.0872,60.2209,61.3427,61.2397,69.3218,-0.2276,-0.8319,0.6042,0.0627,0.9959
3,2020-04-09,AAPL,64.8651,65.1958,63.8995,64.6937,161834800,0.0072,61.3177,60.4594,61.9520,61.5687,69.6954,0.0666,-0.6522,0.7188,0.0583,0.9248
4,2020-04-13,AAPL,64.7710,66.0722,64.1723,65.9635,131022800,0.0196,61.9335,60.4024,62.6813,61.9872,70.7463,0.3976,-0.4422,0.8398,0.0519,0.8233


---
## Summary

- Loaded raw OHLCV data for all 7 tickers.
- Engineered technical indicators: SMA(10,20), EMA(10,20), RSI(14), MACD (+ signal + histogram), Daily Return, and Volatility(20) (+ annualized).
- Dropped warm-up rows so no missing values remain in the feature columns.
- Saved per-ticker feature CSVs and a combined `features_all_tickers.csv` to `data/processed/`.

**Next steps:** merge these features with news sentiment and create the `NextDayUp` target for modeling.